# Task 2: Bag of Words and TF-IDF features

Build representations from the shared seed-46 split. The input joins `Summary` and `Text`; `Score` is not a feature. Fit vocabulary and TF-IDF weights on training data only, then transform validation and test data.


In [1]:
from pathlib import Path
import sys
import pandas as pd

# Works when launched from the repository root or notebooks/.
root = Path.cwd() if (Path.cwd() / "data_utils.py").exists() else Path.cwd().parent
if not (root / "data_utils.py").exists():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory")
sys.path.insert(0, str(root))
from data_utils import load_splits, make_vectorizers

train, val, test = load_splits()
print(f"Rows: train={len(train):,}, validation={len(val):,}, test={len(test):,}")
assert [len(train), len(val), len(test)] == [30123, 6952, 9269]
assert not (set(train.Id) & set(val.Id) or set(train.Id) & set(test.Id) or set(val.Id) & set(test.Id))


Rows: train=30,123, validation=6,952, test=9,269


## Fit on training text only

Both vectorizers come from `data_utils.make_vectorizers()`, so the Naive Bayes script, the error analysis, and the app use exactly these settings:

- **Cleaning** (`clean_text`): lowercase, replace HTML tags such as `<br />` (in about a quarter of reviews) with spaces, and expand `n't` to `not` so negation survives tokenization ("don't" otherwise becomes "don").
- **Unigrams plus bigrams** (`ngram_range=(1, 2)`) to capture phrases such as "not good", "just ok", and "not bad".
- **`min_df=2`**: drop terms that appear in only one training review, which removes most typos and one-off bigrams.
- TF-IDF also uses **sublinear term frequency** (1 + log count) and the default smoothed IDF with L2 row normalization. Bag of Words records raw counts.

No stop words are removed, because words such as "not" and "but" carry sentiment.


In [2]:
bow, tfidf = make_vectorizers()

X_train_bow = bow.fit_transform(train["text"])
X_val_bow = bow.transform(val["text"])
X_test_bow = bow.transform(test["text"])

X_train_tfidf = tfidf.fit_transform(train["text"])
X_val_tfidf = tfidf.transform(val["text"])
X_test_tfidf = tfidf.transform(test["text"])

for name, vectorizer, matrices in [
    ("Bag of Words", bow, (X_train_bow, X_val_bow, X_test_bow)),
    ("TF-IDF", tfidf, (X_train_tfidf, X_val_tfidf, X_test_tfidf)),
]:
    size = len(vectorizer.get_feature_names_out())
    assert all(matrix.shape[1] == size for matrix in matrices)
    bigrams = sum(" " in term for term in vectorizer.get_feature_names_out())
    print(f"{name}: vocabulary={size:,} ({bigrams:,} bigrams); train={matrices[0].shape}, validation={matrices[1].shape}, test={matrices[2].shape}")


Bag of Words: vocabulary=192,646 (173,625 bigrams); train=(30123, 192646), validation=(6952, 192646), test=(9269, 192646)
TF-IDF: vocabulary=192,646 (173,625 bigrams); train=(30123, 192646), validation=(6952, 192646), test=(9269, 192646)
